# (노트) 토픽모델 (R예제와 함께)
> 작성완료

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- categories: [통계계산, 딥러닝, R]

### Toy Example

In [118]:
library(tidyverse)
documents = rbind(c('손흥민','골'),
                  c('골','확률'),
                  c('확률','데이터과학'))

In [119]:
rownames(documents)<-c('doc1','doc2','doc3')
colnames(documents)<-c('word1','word2')
documents

,word1,word2
doc1,손흥민,골
doc2,골,확률
doc3,확률,데이터과학


In [120]:
document_topics=rbind(c(1,0),
                      c(0,1),
                      c(1,0)) ## 랜덤으로 초기화 
rownames(document_topics)<-c('doc1','doc2','doc3')
colnames(document_topics)<-c('topic1','topic2')

In [121]:
document_topics

,topic1,topic2
doc1,1,0
doc2,0,1
doc3,1,0


문서당 토픽비율 
- 문서1: [손흥민, 골] = [1,0]
- 문서2: [골, 확률] = [0,1]
- 문서3: [확률, 데이터과학] = [1,0]

토픽별로 자주 등장하는 단어 
- 토픽0: 골, 골, 데이터과학 ---> 토픽이름은 축구??.. 
- 토픽1: 손흥민, 확률, 확률 ---> 토픽이름은 확률?? 

목표: document_topics의 값을 그럴듯하게 바꾸자. 예를들면 아래처럼. 
```r
document_topics=rbind(doc1=c(0,0),
                      doc2=c(0,1),
                      doc3=c(1,1)) ## 랜덤으로 초기화 
```
그러니까 $3 \times 2$ 차원 랜덤변수를 뽑는문제임. 

--- 

#### 문서1 : [`손흥민`, 골]

`-` 랜덤 초기화로 인한 현재상황은 아래와 같다. 

문서당 토픽비율 
- 문서1: [손흥민, 골] = [1,0]
- 문서2: [골, 확률] = [0,1]
- 문서3: [확률, 데이터과학] = [1,0]

토픽별로 자주 등장하는 단어 
- 토픽0: 골, 골, 데이터과학
- 토픽1: 손흥민, 확률, 확률 

`–` 위에는 임의의 초기값이 들어가있다. 우리는 지금 문서1에서 손흥민에 해당하는 토픽으로 `토픽=0`이 그럴듯한지 `토픽=1`이 그럴듯한지 다시 생각하고 싶다. 

`-` 깁스샘플링: 현재 관심있는 문서1의 첫단어 손흥민에 대한 토픽분류를 제외하고 나머지는 모두 올바른 값이라고 가정하자. 

문서당 토픽비율 
- 문서1: [손흥민, 골] = [1,0] ==> [?,0] 
- 문서2: [골, 확률] = [0,1] ==> [0,1]
- 문서3: [확률, 데이터과학] = [1,0] ==> [1,0]

토픽별로 자주 등장하는 단어 
- 토픽0: 골, 골, 데이터과학 ==> 골, 골, 데이터과학
- 토픽1: 손흥민, 확률, 확률 ==> ?, 확률, 확률

`-` 손흥민이라는 단어가 뽑힐 경우는 아래의 두 가지가 있다. 
 - 토픽0에서 뽑혔을 경우
 - 토픽1에서 뽑혔을 경우

`-` 토픽0에서 뽑혔을 경우는 아래와 같이 계산가능하다 
 - (문서1에 토픽0이 포함되어 있는 비율) * (토픽0에서 손흥민이라는 단어를 뽑을 확률) = 1 * 0.001 
 - p(토픽0|문서1) * p(손흥민|토픽0) = 1 * 0.001

`–` 토픽1에서 뽑혔을경우는 아래와 같이 계산가능하다 
 - p(토픽1|문서1) * p(손흥민|토픽1) = 0.001 * 0.001 

`–` 손흥민은 토픽0에서 뽑혔다고 보는것이 타당함. (왜? 현재단어 손흥민은 문서1에 있고, 문서1은 토픽0이 대부분이니까!) 

`–` 아래와 같이 업데이트 한다. 

문서당 토픽비율 
- 문서1: [손흥민, 골] = [0,0] 
- 문서2: [골, 확률] = [0,1] 
- 문서3: [확률, 데이터과학] = [1,0] 

토픽별로 자주 등장하는 단어 
- 토픽0: 손흥민, 골, 골, 데이터과학
- 토픽1: 확률, 확률 

#### 문서1:[손흥민,`골`]

`-` 업데이트 전

문서당 토픽비율 
- 문서1: [손흥민, 골] = [0,0] ==> [0,?]
- 문서2: [골, 확률] = [0,1] ==> [0,1]
- 문서3: [확률, 데이터과학] = [1,0] ==> [1,0]

토픽별로 자주 등장하는 단어 
- 토픽0: 손흥민, 골, 골, 데이터과학 ==> 손흥민, ?, 골, 데이터과학 
- 토픽1: 확률, 확률 ==> 확률, 확률 

`-` 샘플링 
- p(토픽0|문서1)*p(골|토픽0) = 1 * 1/3
- p(토픽1|문서1)*p(골|토픽1) = 0.001 * 0.001 

결정: 토픽0 선택 (왜? 현재단어는 문서1에 있는데 문서1에는 일단 토픽 0이 대부분이다 `+` 현재 단어는 골인데 골은 토픽0에서 잘나오니까) 

`-` 업데이트 후 

문서당 토픽비율 
- 문서1: [손흥민, 골] = [0,0]
- 문서2: [골, 확률] = [0,1]
- 문서3: [확률, 데이터과학] = [1,0]

---

`-` 이런식으로 반복하면 결국 각 단어는 자신의 정체성을 아래와 같은 2가지 기준으로 판단하게 된다. 

- 나는 $d$-th document에 속해있다 $\to$ 그런데 이 document의 단어를 쭉보니까 토픽 $p$ 에 많다 $\to$ 나도 토픽 $p$ 인가? 
- 나와 똑같은 이름을 가진 단어들이 토픽 $p'$에 많다 $\to$ 나도 토픽 $p'$에서 왔을까? 

`-` 재미있는점은 각 단어의 선택이 다른 단어에도 영향을 준다는 것이다. (다들 나 빼고는 다 맞다고 가정하고 있으니까.. )

--- 

### 구현

`-` 구현해보자. 

In [189]:
data<-read_csv("data.csv")
data


── Column specification ────────────────────────────────────────────────────────
cols(
  word = col_character(),
  doc = col_double(),
  topic = col_double()
)




word,doc,topic
<chr>,<dbl>,<dbl>
교수님,1,1
비대면,1,2
온라인,1,3
코로나,1,1
비대면,1,2
대면,1,3
오프라인,1,1
평가,1,2
비대면,1,3


In [192]:
topic_doc<-function(data,k,d,alpha=0.1){
    count_ = dim(filter(data,doc==d,topic==k))[1] + alpha
    total_ = dim(filter(data,doc==d))[1]+ K*alpha
    count_ / total_ 
}

In [193]:
word_topic<-function(data,w,k,alpha=0.1){
    word_=filter(data,doc==d)[w,]$word
    count_ = dim(filter(data,topic==k,word==word_))[1] + alpha 
    total_ = dim(filter(data,topic==k))[1] + N*alpha 
    count_ / total_ 
}

In [194]:
topicprob<-function(data,d,w){
    rtn<-c()
    for(k in 1:K){
       rtn[k] = topic_doc(data,k,d) * word_topic(data,w,k)
    }
    rtn/sum(rtn)
}

In [195]:
diri<-function(weights){
    cum<- weights %>% cumsum() 
    U<-runif(1)
    1+sum(U>cum)
}

In [196]:
D<-max(data$doc)
K<-3
N<-length(unique(data$word))
for(iter in 1:50){
    i<-0
    for(d in 1:D){
        doclen_ <- dim(filter(data,doc==d))[1]
        for(w in 1:doclen_){
            i<-i+1
            weights_<-topicprob(data[-i,],d,w)
            data[i,]$topic<-diri(weights_)
        }
    }
}
data

word,doc,topic
<chr>,<dbl>,<dbl>
교수님,1,2
비대면,1,2
온라인,1,2
코로나,1,2
비대면,1,2
대면,1,2
오프라인,1,2
평가,1,2
비대면,1,2


In [198]:
data %>% write_csv("results.csv")